In [ ]:
import os
import glob
import numpy as np
import xarray as xr

import matplotlib
from matplotlib import cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1 import make_axes_locatable
%matplotlib inline

from joblib import Parallel, delayed

# --- gridded NetCDF + per-basin river profiles (PyGMT/ArcGIS + matplotlib) ---
from gospl.analyse.gridexport import (
    grid_export, to_netcdf, basin_rivers, plot_long_profile, plot_basin_map)

# --- stratigraphic sections / wells / Wheeler (matplotlib, inline) ---
from gospl.analyse.stratasection import (
    load_strata, cross_section, horizontal_slice, synthetic_well, wheeler,
    well_panel)

## Running the simulation

First activate the conda environment:

```bash
conda activate gospl
```

To run the simulation in a terminal (`X` = number of MPI processes, e.g. 5):

```bash
mpirun -np X gospl -i input-escarpment.yml
```

# Analysing the outputs

All the post-processing below uses goSPL's built-in **`gospl.analyse`** toolkit
(imported in the first cell). Two complementary modules are used:

- **`gospl.analyse.gridexport`** — reassembles the unstructured mesh, rasterises
  every surface field of an output step onto a **regular grid**, runs D8
  hydrology (drainage area, basins, &chi;) and writes a CF-NetCDF for
  PyGMT/ArcGIS. It also exposes per-basin **river long-profile** helpers.
- **`gospl.analyse.stratasection`** — reads the recorded stratigraphy and draws
  **cross-sections, synthetic wells and Wheeler (chronostratigraphic)
  diagrams** (coloured by facies, lithology, provenance, &hellip;).

Every function used below has a **terminal equivalent** so the same products can
be generated outside Jupyter. These console commands &mdash; `gospl-grid`,
`gospl-section`, `gospl-strata-volume` &mdash; are installed with goSPL and are
shown in each section.

Each gridded NetCDF (one file per output step) holds, when available (every
variable carries its `units` and a `long_name` definition):

+ surface elevation `elev` (m) and the step's `sea_level` (m)
+ cumulative erosion/deposition `erodep` (m) and its rate `EDrate` (m/yr)
+ water / sediment fluxes `FA`, `fillFA`, `waterFill`, `sedLoad`
+ hydrology: `drainage_area`, `basin` id, `chi`, `flowdist`, and the
  priority-flood-`filled` elevation

In [ ]:
# Define output folder name for the simulation
out_path = 'results/'

if not os.path.exists(out_path):
    os.makedirs(out_path)

### Rasterising the outputs to a regular grid &mdash; `grid_export` / `to_netcdf`

`grid_export` reassembles the global mesh, interpolates a step's fields onto a
regular grid, runs the D8 hydrology and returns a dict of 2-D arrays;
`to_netcdf` writes that to a CF-NetCDF (each variable annotated with its `units`
and `long_name`). `getOutputs` below simply loops over the steps and writes one
`results/surface<step>.nc` per step.

**`grid_export(h5dir, mesh, step=None, ...)` &mdash; main options**

| Argument | Default | Meaning |
|---|---|---|
| `h5dir` | &ndash; | the run's `h5` output directory |
| `mesh` | &ndash; | global mesh `.npz` (vertices `v`, cells `c`) |
| `step` | last | output step to rasterise |
| `spacing` | median edge | grid resolution `dx[,dy]` (mesh units) |
| `fields` | all | subset of surface fields to include |
| `mn` | `0.5` | &chi; concavity `m/n` |
| `a0` | `1.0` | &chi; reference drainage area |
| `base_level` | run sea level | elevation defining the coast / outlets (catchment + &chi; datum) |
| `latlim` | `89` | (global meshes) crop the polar caps |

Global (spherical) meshes are auto-detected and gridded in lon/lat. The resolved
sea level is stored in each file (global attribute **and** a `sea_level`
variable), so the grid is self-describing.

**Terminal equivalent** (one step &rarr; one NetCDF):

```bash
gospl-grid --h5dir sim_sfd/h5 --mesh data/sombrero.npz:v:c \
    --step 20 --spacing 200 --out results/surface20.nc
```

For the whole time series, loop in the shell:

```bash
for s in $(seq 0 21); do
  gospl-grid --h5dir sim_sfd/h5 --mesh data/sombrero.npz:v:c \
      --step $s --spacing 200 --out results/surface$s.nc
done
```

In [ ]:
h5dir = "sim_sfd/h5"
mesh = "data/sombrero.npz"
reso = 200

out_name = "sfd"

def getOutputs(steps):

    # clear any stale .nc files first
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return
        
    for stp in steps:
        g = grid_export(h5dir, mesh, stp, spacing=reso)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)                       

    return

def getOutputsParallel(steps, h5dir, n_workers=8):
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return

    def process_step(stp):
        g = grid_export(h5dir, mesh, stp, spacing=reso)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)

    Parallel(n_jobs=n_workers)(delayed(process_step)(stp) for stp in steps)

steps = np.arange(21)
getOutputsParallel(steps, h5dir, n_workers=8)
# getOutputs(steps)

In [ ]:
out_name = "mfd"
h5dir = "sim_mfd/h5"
getOutputsParallel(steps, h5dir, n_workers=8)

out_name = "2ngb"
h5dir = "sim_2ngb/h5"
getOutputsParallel(steps, h5dir, n_workers=8)

### Surface elevation through time

The four panels show the remapped `elevation` field at steps 5, 10, 15 and 25, with the black contour marking the $0$ m shoreline. Watch how the coastline migrates as the prescribed sea level and sediment supply reshape the margin: a seaward-stepping shoreline indicates progradation, a landward-stepping one indicates transgression.

In [ ]:
sim1 = xr.open_dataset("results/sfd20.nc")
sim2 = xr.open_dataset("results/2ngb20.nc")
sim3 = xr.open_dataset("results/mfd20.nc")

We map the flow discharge (`flowDischarge`, m3/yr, masked to land where elevation > 0) on a log colour scale for each scheme, with a zoom on a 20 x 20 km window centred at (30, 50) km. SFD concentrates all flow into single channels, giving thin, sharply defined high-discharge threads; MFD spreads discharge across neighbouring nodes, producing broader, more diffuse drainage with smoother accumulation gradients.

Run the following data preparation or mesh generation step.

The same discharge comparison is repeated for a second zoom window at (5, 50) km, sampling a different part of the radial drainage system. Comparing the two regions confirms that the SFD branching pattern shifts with node placement, whereas the MFD discharge field is more stable and less tied to the mesh geometry.

In [ ]:
geometries = [
    {
        'type': 'Polygon',
        'coordinates': [[
            [30000., 50000.],
            [30000., 70000.],
            [50000., 70000.],
            [50000., 50000.]
        ]]
    }
]


fig, ax = plt.subplots(1,2, figsize=(8,4))
sim1.rio.write_crs('epsg:4326', inplace=True)
im = sim1.elev.plot(ax=ax[0], add_labels=False, add_colorbar=False, vmin=-100, vmax=600, cmap='Spectral_r')
# Create a Rectangle patch
rect = patches.Rectangle((30000, 50000), 20000, 20000, linewidth=1, edgecolor='k', facecolor='none')
# Add the patch to the Axes
ax[0].add_patch(rect)

ax[0].set_title('SFD', fontsize=10, fontweight="bold")

# Clip your region
clipped = sim1.rio.clip(geometries)
clipped.elev.plot(ax=ax[1], add_labels=False, add_colorbar=False, vmin=-100, vmax=600, cmap='Spectral_r')

ax[1].set_title('SFD zoom', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.04, 0.6, 0.04]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Elevation (m)')

# plt.tight_layout()
plt.show()

################################

fig, ax = plt.subplots(1,2, figsize=(8,4))
sim2.rio.write_crs('epsg:4326', inplace=True)
im = sim2.elev.plot(ax=ax[0], add_labels=False, add_colorbar=False, vmin=-100, vmax=600, cmap='Spectral_r')
# Create a Rectangle patch
rect = patches.Rectangle((30000, 50000), 20000, 20000, linewidth=1, edgecolor='k', facecolor='none')
# Add the patch to the Axes
ax[0].add_patch(rect)

ax[0].set_title('2 downstream flow direction', fontsize=10, fontweight="bold")

# Clip your region
clipped = sim2.rio.clip(geometries)
clipped.elev.plot(ax=ax[1], add_labels=False, add_colorbar=False, vmin=-100, vmax=600, cmap='Spectral_r')

ax[1].set_title('2 downstream flow direction zoom', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.04, 0.6, 0.04]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Elevation (m)')

# plt.tight_layout()
plt.show()

################################

fig, ax = plt.subplots(1,2, figsize=(8,4))
sim3.rio.write_crs('epsg:4326', inplace=True)
im = sim3.elev.plot(ax=ax[0], add_labels=False, add_colorbar=False, vmin=-100, vmax=600, cmap='Spectral_r')
# Create a Rectangle patch
rect = patches.Rectangle((30000, 50000), 20000, 20000, linewidth=1, edgecolor='k', facecolor='none')
# Add the patch to the Axes
ax[0].add_patch(rect)

ax[0].set_title('MFD', fontsize=10, fontweight="bold")

# Clip your region
clipped = sim3.rio.clip(geometries)
clipped.elev.plot(ax=ax[1], add_labels=False, add_colorbar=False, vmin=-100, vmax=600, cmap='Spectral_r')

ax[1].set_title('MFD zoom', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.04, 0.6, 0.04]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Elevation (m)')

# plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,5))
im = (sim2.elev-sim1.elev).plot(ax=ax[0], add_labels=False, add_colorbar=False, vmin=-10, vmax=10, cmap='bwr')
ax[0].set_title('2 downstream flow - SFD', fontsize=10, fontweight="bold")

(sim3.elev-sim1.elev).plot(ax=ax[1], add_labels=False, add_colorbar=False, vmin=-10, vmax=10, cmap='bwr')
ax[1].set_title('MFD - SFD', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.04, 0.6, 0.04]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Elevation difference (m)')

# plt.tight_layout()
plt.show()

## Flow discharge

Let's look at the flow discharge btw the different experiments.

In [ ]:
fa1 = xr.open_dataset("results/sfd2.nc")
nfa1 = fa1.where(fa1.elev>0)
fa2 = xr.open_dataset("results/2ngb2.nc")
nfa2 = fa2.where(fa1.elev>0)
fa3 = xr.open_dataset("results/mfd2.nc")
nfa3 = fa3.where(fa3.elev>0)

Run the following data preparation or mesh generation step.

In [ ]:
geometries = [
    {
        'type': 'Polygon',
        'coordinates': [[
            [30000., 50000.],
            [30000., 70000.],
            [50000., 70000.],
            [50000., 50000.]
        ]]
    }
]

fig, ax = plt.subplots(1,2, figsize=(9,4))
nfa1.rio.write_crs('epsg:4326', inplace=True)
im = nfa1.FA.plot(ax=ax[0], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')
# Create a Rectangle patch
rect = patches.Rectangle((30000, 50000), 20000, 20000, linewidth=1, edgecolor='k', facecolor='none')
# Add the patch to the Axes
ax[0].add_patch(rect)

ax[0].set_title('SFD', fontsize=10, fontweight="bold")

# Clip your region
clipped = nfa1.rio.clip(geometries)
clipped.FA.plot(ax=ax[1], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')

ax[1].set_title('SFD zoom', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.04, 0.6, 0.04]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Flow discharge (m3/yr)')

# plt.tight_layout()
plt.show()

################################

fig, ax = plt.subplots(1,2, figsize=(9,4))
nfa2.rio.write_crs('epsg:4326', inplace=True)
im = nfa2.FA.plot(ax=ax[0], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')
# Create a Rectangle patch
rect = patches.Rectangle((30000, 50000), 20000, 20000, linewidth=1, edgecolor='k', facecolor='none')
# Add the patch to the Axes
ax[0].add_patch(rect)

ax[0].set_title('2 downstream flow', fontsize=10, fontweight="bold")

# Clip your region
clipped = nfa2.rio.clip(geometries)
clipped.FA.plot(ax=ax[1], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')

ax[1].set_title('2 downstream flow zoom', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.04, 0.6, 0.04]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Flow discharge (m3/yr)')

# plt.tight_layout()
plt.show()

################################

fig, ax = plt.subplots(1,2, figsize=(9,4))
nfa3.rio.write_crs('epsg:4326', inplace=True)
im = nfa3.FA.plot(ax=ax[0], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')
# Create a Rectangle patch
rect = patches.Rectangle((30000, 50000), 20000, 20000, linewidth=1, edgecolor='k', facecolor='none')
# Add the patch to the Axes
ax[0].add_patch(rect)

ax[0].set_title('MFD', fontsize=10, fontweight="bold")

# Clip your region
clipped = nfa3.rio.clip(geometries)
clipped.FA.plot(ax=ax[1], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')

ax[1].set_title('MFD zoom', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.04, 0.6, 0.04]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Flow discharge (m3/yr)')

# plt.tight_layout()
plt.show()

Run the following data preparation or mesh generation step.


In [ ]:
geometries = [
    {
        'type': 'Polygon',
        'coordinates': [[
            [5000., 50000.],
            [5000., 70000.],
            [25000., 70000.],
            [25000., 50000.]
        ]]
    }
]

fig, ax = plt.subplots(1,2, figsize=(9,4))
nfa1.rio.write_crs('epsg:4326', inplace=True)
im = nfa1.FA.plot(ax=ax[0], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')
# Create a Rectangle patch
rect = patches.Rectangle((5000, 50000), 20000, 20000, linewidth=1, edgecolor='k', facecolor='none')
# Add the patch to the Axes
ax[0].add_patch(rect)

ax[0].set_title('SFD', fontsize=10, fontweight="bold")

# Clip your region
clipped = nfa1.rio.clip(geometries)
clipped.FA.plot(ax=ax[1], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')

ax[1].set_title('SFD zoom', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.04, 0.6, 0.04]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Flow discharge (m3/yr)')

# plt.tight_layout()
plt.show()

################################

fig, ax = plt.subplots(1,2, figsize=(9,4))
nfa2.rio.write_crs('epsg:4326', inplace=True)
im = nfa2.FA.plot(ax=ax[0], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')
# Create a Rectangle patch
rect = patches.Rectangle((5000, 50000), 20000, 20000, linewidth=1, edgecolor='k', facecolor='none')
# Add the patch to the Axes
ax[0].add_patch(rect)

ax[0].set_title('2 downstream flow', fontsize=10, fontweight="bold")

# Clip your region
clipped = nfa2.rio.clip(geometries)
clipped.FA.plot(ax=ax[1], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')

ax[1].set_title('2 downstream flow zoom', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.04, 0.6, 0.04]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Flow discharge (m3/yr)')

# plt.tight_layout()
plt.show()

################################

fig, ax = plt.subplots(1,2, figsize=(9,4))
nfa3.rio.write_crs('epsg:4326', inplace=True)
im = nfa3.FA.plot(ax=ax[0], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')
# Create a Rectangle patch
rect = patches.Rectangle((5000, 50000), 20000, 20000, linewidth=1, edgecolor='k', facecolor='none')
# Add the patch to the Axes
ax[0].add_patch(rect)

ax[0].set_title('MFD', fontsize=10, fontweight="bold")

# Clip your region
clipped = nfa3.rio.clip(geometries)
clipped.FA.plot(ax=ax[1], add_labels=False, add_colorbar=False, norm=matplotlib.colors.LogNorm(vmin=1e5, vmax=1e8),cmap='Blues')

ax[1].set_title('MFD zoom', fontsize=10, fontweight="bold")

cbar_ax = fig.add_axes([0.2, -0.04, 0.6, 0.04]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Flow discharge (m3/yr)')

# plt.tight_layout()
plt.show()

After 10 000 years, the dendritic flow accumulation pattern observed on the surface for the SFD case is analogue to many natural forms of drainage systems but is actually a numerical artefact and depends on the random locations of the nodes in the surface triangulation. 

By increasing the number of possible downstream directions, this sensitivity to the mesh discretisation is significantly reduced. In addition, routing flow to more than one destination node allows for a better representation of channel pathway divergence into multiple branches over flat regions.

Landscape evolution models tend to be highly dependent on grid resolution, and this dependency is mostly related to the approach used to route water down the surface. 

Enabling the node-to-node MFD algorithm decreases the dependence of landscape features (e.g. valley spacing, branching of stream network, sediment flux) on grid resolution. As shown in the above figures, the SFD algorithm leads to increased branching of valleys, whereas the MFD approach, by promoting wider flow distribution, produces smoother topography on which local carving of the landscape is reduced.It has also been shown that when using models that operate at a scale larger than river-width resolution, the node-to-node MFD algorithm creates landscape features that are not resolution dependent and that evolve closer to the ones observed in nature. Therefore, it is recommended to use more than one downhill direction (flowdir) in goSPL when looking at global- and continental-scale landscape evolution or for cases in which multiple resolutions are considered within a given mesh.